# 02C SCD1 and SCD2 Demo

Load complete Product Master snapshots into one Bronze source, then publish the same governed Full Read to SCD1 and SCD2 Silver targets.


## Before you begin

- Import this notebook beside `00_env_config` and attach the same Fabric Environment.
- Upload `templates/DemoData` to the Bronze Lakehouse `Files/Demo` location.
- Run Day 1, then Day 2, then Day 3. Do not run an earlier day after a later day.
- Re-running the current day is replay-safe; `scd2` does not create a new version when tracked values are unchanged.


# 0. Environment


In [ ]:
%run 00_env_config


In [ ]:
from pyspark.sql import functions as F

from fabricops_kit import (
    orchestrate_read,
    orchestrate_write,
    read_lakehouse_csv,
    read_lakehouse_table,
    widget_select_data_contract,
    write_lakehouse_table,
)


# 1. Choose the demo day

Start with `1`. After a successful complete run, change only this value to `2` and rerun from the top, then repeat with `3`.


In [ ]:
DEMO_DAY = 1

if DEMO_DAY not in {1, 2, 3}:
    raise ValueError("DEMO_DAY must be 1, 2, or 3.")


# 2. Read the selected snapshot and overwrite Bronze


In [ ]:
snapshot_df = read_lakehouse_csv(
    f"Demo/scd_product_master/products_day{DEMO_DAY}.csv",
    store="Bronze",
    spark_session=spark,
    header=True,
    inferSchema=True,
)

if snapshot_df.count() != 8:
    raise ValueError(f"Day {DEMO_DAY} must contain exactly 8 Product Master rows.")


In [ ]:
write_lakehouse_table(
    snapshot_df,
    table_name="product_master_updates",
    store="Bronze",
    schema="demo",
    mode="overwrite",
)


# 3. Data Contract

Select contracts when they exist. To publish both SCD targets, leave them unselected in Development or select **Enforce**; **Validate** evaluates without publishing.


In [ ]:
CONTRACTS = widget_select_data_contract(spark_session=spark)


# 4. Governed Full Read


In [ ]:
sources = {}


In [ ]:
source = orchestrate_read(
    name="product_master",
    store="Bronze",
    schema="demo",
    table_name="product_master_updates",
    read_mode="full",
    query=None,
    spark_session=spark,
)
sources["product_master"] = source


# 5. PySpark Transform


In [ ]:
transformed_df = (
    sources["product_master"]["dataframe"]
    .select(
        "product_id",
        "product_name",
        "product_category",
        F.col("list_price").cast("double").alias("list_price"),
        F.col("modified_datetime").cast("timestamp").alias("modified_datetime"),
    )
)


## Sequence safety

For Days 2 and 3, verify that the prior SCD2 state exists before publishing. This catches skipped or out-of-order days instead of silently rewriting history.


In [ ]:
if DEMO_DAY > 1:
    existing_scd2_df = read_lakehouse_table(
        table_name="product_master_scd2",
        store="Silver",
        schema="demo",
        spark_session=spark,
    )
    allowed_existing_versions = {2: {8, 9}, 3: {9, 10}}[DEMO_DAY]
    actual_existing_versions = existing_scd2_df.count()
    if actual_existing_versions not in allowed_existing_versions:
        raise ValueError(
            f"Day {DEMO_DAY} requires an existing SCD2 version count in "
            f"{sorted(allowed_existing_versions)}; "
            f"found {actual_existing_versions}. Reset or run the missing prior day first."
        )


# 6. Governed SCD Writes

Both targets receive the same transformed DataFrame and governed Read result. Each Write is an independent publication.


In [ ]:
writes = {}


## WRITE 1 — SCD1 current state


In [ ]:
writes["product_master_scd1"] = orchestrate_write(
    transformed_df,
    name="product_master_scd1",
    sources=[sources["product_master"]],
    store="Silver",
    schema="demo",
    table_name="product_master_scd1",
    write_mode="scd1",
    write_parameters={"key_columns": ["product_id"]},
    contracts=CONTRACTS,
    repartition_by=None,
    spark_session=spark,
)


## WRITE 2 — SCD2 history


In [ ]:
writes["product_master_scd2"] = orchestrate_write(
    transformed_df,
    name="product_master_scd2",
    sources=[sources["product_master"]],
    store="Silver",
    schema="demo",
    table_name="product_master_scd2",
    write_mode="scd2",
    write_parameters={
        "key_columns": ["product_id"],
        "effective_column": "modified_datetime",
        "tracked_columns": ["product_category", "list_price"],
    },
    contracts=CONTRACTS,
    repartition_by=None,
    spark_session=spark,
)


# 7. Inspect current and historical results


In [ ]:
scd1_df = read_lakehouse_table(
    table_name="product_master_scd1",
    store="Silver",
    schema="demo",
    spark_session=spark,
)
scd2_df = read_lakehouse_table(
    table_name="product_master_scd2",
    store="Silver",
    schema="demo",
    spark_session=spark,
)

print("SCD1 rows:", scd1_df.count())
print("SCD2 versions:", scd2_df.count())
display(scd1_df.orderBy("product_id"))
display(
    scd2_df.select(
        "product_id",
        "product_name",
        "product_category",
        "list_price",
        "modified_datetime",
        "_effective_from",
        "_effective_to",
        "_is_current",
    ).orderBy("product_id", "_effective_from")
)


## Expected results and reset

| Day | SCD1 rows | SCD2 versions |
|---:|---:|---:|
| 1 | 8 | 8 |
| 2 | 8 | 9 |
| 3 | 8 | 10 |

For a clean reset, remove the Bronze `demo.product_master_updates` table and both Silver demo targets, then set `DEMO_DAY = 1`. If contracts or other governed metadata were authored specifically for these demo identities, reset them only through the normal FabricOps Governance workflow or use a fresh Guided Demo environment. Never restart at Day 1 against retained SCD targets.
